# APTOS ordinal-versus-MSE research execution

This Kaggle-ready notebook checks the environment and constructs the commands. **Training is disabled by default**. Set `RUN_EXPERIMENTS=True` only in your authorized GPU notebook after preparing the image cache and accepting the compute cost. Results must not be invented or optimized using the held-out test set.


In [ ]:
import subprocess
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / 'configs' / 'cv.yaml').exists():
    raise FileNotFoundError('Start inside the checked-out APTOS research repository')
RUN_EXPERIMENTS = False  # Explicit opt-in before consuming Kaggle GPU time
SEEDS = (42, 43, 44)
print('Python:', sys.version.split()[0], 'repository:', ROOT)


## Experimental matrix

Both modes share identical APTOS folds, preprocessing and seeds. Each full five-fold run can be expensive; GPU and authorized Kaggle data are prerequisites.


In [ ]:
matrix = [[sys.executable, '-m', 'aptos.training.cv', '--config', 'configs/cv.yaml', '--mode', mode, '--seed', str(seed)]
          for seed in SEEDS for mode in ('reg', 'ordinal')]
for cmd in matrix:
    print(' '.join(cmd))
if RUN_EXPERIMENTS:
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError('GPU unavailable: refusing to start full experiment matrix')
    for cmd in matrix:
        subprocess.run(cmd, check=True, cwd=ROOT)
else:
    print('Dry run only. No training has been executed.')


## After training

Record exact sweep directory names from each completed run. Score both models on IDRiD and Messidor-2 with the frozen APTOS thresholds, then run `python -m aptos.evaluation.compare_sweeps` for each cohort. See [benchmark protocol](../docs/ordinal-benchmark.md) for commands. Do not interpret missing outputs as model failure; they indicate missing execution.
